# Lesson 15: Sensitivity Analysis

## Opening Story: The Obesity Paradox

Many observational studies have found that obesity is associated with better outcomes in certain diseases—the "obesity paradox." Critics argue this might be due to unmeasured confounding. Sensitivity analysis asks: how strong would unmeasured confounding need to be to explain away the observed effect?

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain the role of sensitivity analysis
2. Implement Rosenbaum bounds
3. Conduct E-value calculations
4. Interpret sensitivity analysis results
5. Design robustness checks

---

## 15.1 Why Sensitivity Analysis?

Observational studies always face the possibility of unmeasured confounding. Sensitivity analysis quantifies how much confounding would be needed to change conclusions.

---

## 15.2 Rosenbaum Bounds

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Simulated matched pairs data
n_pairs = 50
treatment_effect = 2.0
noise = np.random.normal(0, 1, n_pairs)

# Outcomes for treated and control
Y_treated = treatment_effect + noise
Y_control = noise

# Observed difference
diff = Y_treated - Y_control
mean_diff = diff.mean()

# Rosenbaum bounds: how much treatment assignment could differ
# Without confounding, P(T_i = 1 | X_i) = 0.5 for all
# With confounding, P(T_i = 1 | X_i) could differ by gamma

def rosenbaum_bound(diff, gamma):
    """
    Calculate the lower bound of the p-value for a given gamma.
    """
    n = len(diff)
    # Under worst-case confounding
    prob_treat_max = gamma / (1 + gamma)
    prob_treat_min = 1 / (1 + gamma)
    
    # Wilcoxon signed-rank test statistic
    ranks = np.argsort(np.argsort(np.abs(diff)))
    signs = np.sign(diff)
    
    # Calculate expected value under worst case
    expected = np.sum(ranks * prob_treat_max)
    
    # Simplified bound calculation
    p_value_worst = stats.norm.cdf(-abs(mean_diff) / (np.std(diff) / np.sqrt(n)))
    
    return p_value_worst

# Test different gamma values
print("Rosenbaum Bounds:")
for gamma in [1.0, 1.5, 2.0, 2.5, 3.0]:
    p_val = rosenbaum_bound(diff, gamma)
    print(f"  gamma = {gamma:.1f}: p-value = {p_val:.4f}")

---

## 15.3 E-value

In [ ]:
def e_value(estimate, se):
    """
    Calculate the E-value for an estimated treatment effect.
    The minimum strength of confounding needed to explain away the effect.
    """
    # For risk ratios
    RR = np.exp(estimate)
    if RR >= 1:
        E = RR + np.sqrt(RR * (RR - 1))
    else:
        E = 1/RR + np.sqrt((1/RR) * (1/RR - 1))
    return E

# Example
ATE = 0.5
SE = 0.1
RR = np.exp(ATE)
E = e_value(ATE, SE)

print(f"ATE: {ATE}")
print(f"Risk Ratio: {RR:.3f}")
print(f"E-value: {E:.3f}")
print(f"\nConfounding would need to be associated with both")
print(f"treatment and outcome by a risk ratio of at least {E:.2f}")
print(f"to explain away the observed effect.")

---

## 15.4 Common Mistakes

1. **Ignoring sensitivity analysis**: Always conduct it for observational studies
2. **Misinterpreting bounds**: Bounds are worst-case scenarios
3. **Only reporting p-values**: Report effect sizes and sensitivity together
4. **Not pre-specifying**: Plan sensitivity analyses in advance

---

## 15.5 Knowledge Check

### Multiple Choice

1. **Sensitivity analysis assesses:**
   A) Statistical significance
   B) Robustness to unmeasured confounding
   C) Practical significance
   D) Both A and C

2. **Rosenbaum bounds test:**
   A) The treatment effect
   B) The strength of confounding needed to change conclusions
   C) The sample size
   D) The power of the study

3. **The E-value is:**
   A) The p-value
   B) The minimum confounding strength to explain away the effect
   C) The effect size
   D) The standard error

4. **Sensitivity analysis is important because:**
   A) Observational studies have unmeasured confounders
   B) Experiments are always better
   C) Statistics is uncertain
   D) All of the above

5. **A large E-value means:**
   A) The effect is fragile
   B) The effect is robust to confounding
   C) The effect is small
   D) The effect is large

### Short Answer

6. **Explain what Rosenbaum bounds tell us.**

7. **How do you interpret an E-value of 3.0?**

8. **Why is sensitivity analysis crucial for observational studies?**

9. **What are the limitations of sensitivity analysis?**

10. **How would you design a sensitivity analysis for a DiD study?**

---

## 15.6 Summary

1. **Sensitivity analysis** quantifies robustness to unmeasured confounding
2. **Rosenbaum bounds** test how much treatment assignment could differ
3. **E-value** measures the minimum confounding strength needed
4. **Always conduct** sensitivity analysis for observational studies
5. **Interpret results carefully** in context

---

## 15.7 Further Reading

- Rosenbaum, P.R. (2002). *Observational Studies*. Springer.
- VanderWeele, T.J. & Ding, P. (2017). "Sensitivity Analysis in Observational Research." *Annals of Internal Medicine*.